# 12 · Review of the LLM NE gate: gated vs ungated NE-positive IHC notes

Notebook 11 calls a qualifying IHC note **NE-positive** when its report has ≥1 LLM NEPC finding that passes LCA's
`gate_findings`. That gate runs the trial-context screen, then `ne_gate.screen_ne_quote` (ne_gate v3). This notebook
asks how much that gate decides the NE outcomes. It covers the false positives the gate lets through, the true
positives it drops, and the NE-marker notes the LLM never flagged. All logic lives in
`COMPASS/multivariable_models/ihc_gate_review.py`. LCA is imported read-only.

**Scope.** Every qualifying IHC note (nb11's regex and specimen rule) of a patient 04 scored, at **any date**.
The sheet's `in_window_<landmark>` columns flag notes that fall in nb11's future window at that landmark.

**Review groups** (one per note):

| Group | Meaning | On the sheet |
|---|---|---|
| `gated` | ≥1 finding passes the full gate (nb11 NE-positive) | yes, as the comparison arm (gate precision) |
| `gate_rejected` | ≥1 finding survives the trial-context screen, but ne_gate rejects all of them ("ungated-only" positive) | yes |
| `no_finding` | NE marker in the report; the LLM read it and returned no finding | yes |
| `not_run_or_failed` | NE marker; in the LLM evidence, but the final attempt is missing or failed | yes |
| `not_in_evidence` | NE marker; never selected into the LLM evidence snippets | yes |
| `trial_context_only` | every finding was dropped by the trial-context screen | counted only |

Notes with no finding and no NE marker get no group.

**"Ungated"** removes ne_gate only. The trial-context screen and final-attempt compaction stay on, as
`load_label_inputs` applies them. Each finding records the step and reason that kept or dropped it.
The audit checks that the findings marked `passed` match `gate_findings` exactly and stops if they differ.

**Gated vs ungated.** For each definition, the notebook reports nb11's `ne_pos_all` / `ne_pos_given_ihc` n and
events per landmark. It uses nb11's merge, frame and populations, and **nothing is refit**.

**Manual review workflow.**
1. Run through *Write outputs*. This writes `gate_review_sheet.csv`, which has blank `reviewer_call` and
   `reviewer_note` columns.
2. Save a copy as `gate_review_sheet_reviewed.csv` in the same folder. Fill `reviewer_call` with `true_ne`,
   `not_ne` or `unclear`, or leave it blank for not reviewed.
3. Re-run the last cell. It reports the true-NE fraction per group and per ne_gate reason (Wilson 95% CI), and
   the event counts under a third, `reviewed`, definition. That definition starts from the gated notes, drops
   those called `not_ne` and adds any note called `true_ne`.

The script refuses to overwrite a `gate_review_sheet.csv` that already has calls. Review IDs are
`DFCI_MRN:RPT_ID`. Rows in the reviewed copy that are no longer on the sheet are reported as stale.

**Reading the sheet.** The LLM saw only the evidence snippet, not the full report. `text_window` shows the full
report around NE terms and NE markers (±`WINDOW_CHARS` characters), and `ne_marker_in_snippet` says whether a
marker made it into the snippet.

**PHI.** The sheet and the findings audit hold report text and LLM quotes. They are written only under the
protected `DATA_ROOT/ihc_lr/gate_review/`.

**Prerequisites.** These are the same as notebook 11.

In [ ]:
import json, os, sys
sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp
for _p in (str(cp.PROJECT_ROOT),):
    if _p not in sys.path: sys.path.insert(0, _p)
from pathlib import Path
import pandas as pd
from IPython.display import display
from COMPASS.multivariable_models import ihc_lr as il
from COMPASS.multivariable_models import ihc_gate_review as igr
nsl = il.nsl

DATA_ROOT  = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
LANDMARKS      = (0, 90, 180)
WINDOW_CHARS   = igr.DEFAULT_WINDOW_CHARS     # characters either side of each NE term / marker hit
MAX_TEXT_CHARS = igr.DEFAULT_MAX_TEXT_CHARS   # cap on text_window per note

SCORES_PATH    = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"
DATA_PATH      = DATA_ROOT / "longitudinal_prediction_data_adt.csv"      # ADT anchor dates
PATHOLOGY_PATH = Path(os.environ.get("PROFILE_PATHOLOGY_NOTES",
                                     str(Path(cp._PROFILE_DATA_ROOT) / "CLINICAL_NOTES" / "PATHOLOGY_NOTES.parquet")))
LLM_NEPC_DIR   = Path("/data/gusev/USERS/jpconnor/data/LLM_annotations/LLM_nepc")
# LLM_clinical_annotations: a sibling of the project (cluster: /data/gusev/USERS/jpconnor/code/), else one level up (local).
_LCA_CANDIDATES = [Path(cp.PROJECT_ROOT).parent / "LLM_clinical_annotations",
                   Path(cp.PROJECT_ROOT).parent.parent / "LLM_clinical_annotations"]
LCA_ROOT       = Path(os.environ.get("COMPASS_LCA_ROOT",
                                     str(next((p for p in _LCA_CANDIDATES if p.exists()), _LCA_CANDIDATES[0]))))
OUT_DIR        = DATA_ROOT / "ihc_lr" / "gate_review"   # protected data root; PHI (quotes, report text)
SHEET_PATH     = OUT_DIR / "gate_review_sheet.csv"
REVIEWED_PATH  = OUT_DIR / "gate_review_sheet_reviewed.csv"

## Load inputs

In [ ]:
scores = nsl.load_transfer_scores(SCORES_PATH, family="xgboost_cox", config="both", endpoint="nepc")
scores = scores[scores["landmark_days"].isin(LANDMARKS)]
anchors = il.load_anchor_dates(DATA_PATH)
scored_ids = set(scores[il.ID_COL].astype(int))
il.validate_anchor_coverage(scores, anchors)

lca = il.import_lca(LCA_ROOT)     # read-only import; nothing in LCA is written
pats = il.build_patterns(lca["ne_gate"].NE_MARKER.pattern)
notes = il.load_pathology_notes(PATHOLOGY_PATH, mrns=scored_ids)   # text kept: the sheet needs it
ihc_notes = il.tag_ihc_notes(notes, pats)
print(f"{len(scored_ids):,} scored patients; {len(notes):,} pathology reports; "
      f"{int(ihc_notes['qualifies'].sum()):,} qualifying IHC notes")
print(f"LCA {il.lca_commit(LCA_ROOT)[:12]} · {lca['ne_gate'].NE_GATE_VERSION} · {il.IHC_REGEX_VERSION}")

## Audit every LLM finding and sort the qualifying IHC notes into review groups

`findings` has one row per final-attempt pathology finding of a scored patient. `gate_stage` is `passed`,
`ne_gate` or `trial_context`, and `gate_reason` is the screen's reason code. The reason table counts findings
on every pathology report, whatever the IHC regex says. The group table counts qualifying IHC notes.

In [ ]:
inputs, ledger = igr.load_llm_inputs(lca, LLM_NEPC_DIR)
findings = igr.finding_audit(lca, inputs, mrns=scored_ids)        # stops if it disagrees with gate_findings
llm_notes = igr.llm_note_table(inputs, ledger, mrns=scored_ids)   # stops on ambiguous report IDs
review_status = {int(m): s for m, s in inputs["status"].items() if int(m) in scored_ids}
classified = igr.classify_notes(ihc_notes, findings, llm_notes, pats,
                                ok_statuses=lca["note_extraction"].OK_STATUSES)

print(f"{len(findings):,} pathology findings for scored patients "
      f"({findings['gate_stage'].value_counts().to_dict()})")
print("=== Findings by gate step and reason × evidence type (all pathology reports) ===")
display(igr.reason_table(findings, classified))
print("=== Qualifying IHC notes by review group ===")
display(igr.group_counts(classified))

## Gated vs ungated: nb11 NE-outcome counts (no refit)

`gated` reproduces nb11. `ungated` also counts reports whose findings ne_gate rejected. The patient population and
review-status rule are nb11's. Only the set of NE-positive reports changes.

In [ ]:
definitions = igr.definition_keys(findings)
events = igr.event_counts(scores, ihc_notes, definitions, review_status, anchors, LANDMARKS)
display(events.pivot_table(index=["landmark_days", "outcome"], columns="definition",
                           values=["n", "events"], aggfunc="first"))

## Write outputs

In [ ]:
sheet = igr.build_review_sheet(classified, notes, scores, anchors, LANDMARKS, igr.review_pattern(lca["ne_gate"]),
                               window=WINDOW_CHARS, max_chars=MAX_TEXT_CHARS)
igr.write_review_sheet(sheet, SHEET_PATH)
findings.to_parquet(OUT_DIR / "gate_findings_audit.parquet", index=False)
igr.group_counts(classified).to_csv(OUT_DIR / "gate_review_groups.csv", index=False)
events.to_csv(OUT_DIR / "gate_event_counts.csv", index=False)
meta = {
    "lca_root": str(LCA_ROOT), "lca_commit": il.lca_commit(LCA_ROOT),
    "ne_gate_version": lca["ne_gate"].NE_GATE_VERSION, "ihc_regex_version": il.IHC_REGEX_VERSION,
    "ungated_definition": "ne_gate removed; trial-context screen and final-attempt compaction kept",
    "note_scope": "qualifying IHC notes of scored patients, any date",
    "review_groups": list(igr.REVIEW_GROUPS), "unreviewed_groups": list(igr.UNREVIEWED_GROUPS),
    "reviewer_calls": list(igr.REVIEWER_CALLS),
    "window_chars": WINDOW_CHARS, "max_text_chars": MAX_TEXT_CHARS,
    "scores_path": str(SCORES_PATH), "pathology_path": str(PATHOLOGY_PATH), "llm_nepc_dir": str(LLM_NEPC_DIR),
    "landmarks": list(LANDMARKS), "max_followup_days": il.MAX_FOLLOWUP_DAYS, "sheet_rows": len(sheet),
}
(OUT_DIR / "run_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Wrote {SHEET_PATH.name} ({len(sheet):,} notes: {sheet['review_group'].value_counts().to_dict()}), "
      f"gate_findings_audit.parquet, gate_review_groups.csv, gate_event_counts.csv, run_meta.json to {OUT_DIR}")

## Read back the manual review

This cell runs only when `gate_review_sheet_reviewed.csv` exists. `frac_true_ne` = true_ne / (true_ne + not_ne),
which leaves out `unclear`, with a Wilson 95% CI. For `gated` it is the gate's precision. For the other groups it
is the share of real NE positives the gate or the LLM missed. The by-reason table splits `gate_rejected` notes by
each ne_gate reason code on the note.

In [ ]:
if REVIEWED_PATH.exists():
    reviewed, stale = igr.read_reviewed(REVIEWED_PATH, sheet)
    if stale:
        print(f"WARNING: {len(stale)} reviewed rows are no longer on the sheet (ignored): {stale[:10]}")
    print(f"{reviewed['reviewer_call'].ne('').sum():,} of {len(reviewed):,} sheet rows reviewed")
    print("=== Reviewer calls by review group ===")
    display(igr.summarize_review(reviewed).round(3))
    print("=== gate_rejected notes by ne_gate reason ===")
    display(igr.summarize_by_reason(reviewed).round(3))
    rev_events = igr.event_counts(scores, ihc_notes, igr.definition_keys(findings, reviewed),
                                  review_status, anchors, LANDMARKS)
    print("=== nb11 NE-outcome counts: gated vs ungated vs reviewed (no refit) ===")
    display(rev_events.pivot_table(index=["landmark_days", "outcome"], columns="definition",
                                   values=["n", "events"], aggfunc="first"))
else:
    print(f"No reviewed sheet yet. Copy {SHEET_PATH} to {REVIEWED_PATH.name} in the same folder, "
          f"fill reviewer_call ({' / '.join(igr.REVIEWER_CALLS)}, blank = not reviewed), then re-run this cell.")